# Gemma 4 Audio Reasoning

This notebook loads Gemma 4 E4B and sends an audio + text prompt with reasoning enabled.

Run the setup cell first. The current environment has an older `transformers` install that is incompatible with the installed `huggingface_hub`, so the upgrade is required before loading the model.

Replace `AUDIO_PATH` with a real clip before using this for analysis. `data/silence.wav` is only a placeholder smoke test file.

In [1]:
# Run this once, then restart the kernel so the upgraded packages are picked up.
%pip install -q transformers librosa accelerate soundfile datasets scikit-learn pandas pyarrow scipy seaborn matplotlib einops git+https://github.com/davidbau/baukit

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "/project/jevans/tzhang3/models/gemma-4-E4B-it"
# Alternative:
# MODEL_ID = "google/gemma-4-E4B-it"

AUDIO_SOURCE = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/journal1.wav"
USER_PROMPT = (
    "Listen carefully to the audio. First transcribe any speech in its original "
    "language. Then briefly explain the main sounds you hear and your confidence."
)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": AUDIO_SOURCE},
            {"type": "text", "text": USER_PROMPT},
        ],
    }
]

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)
model.eval()

print(f"Using model: {MODEL_ID}")
print(f"Audio source: {AUDIO_SOURCE}")

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Using model: /project/jevans/tzhang3/models/gemma-4-E4B-it
Audio source: https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/journal1.wav


In [3]:
def run_audio_prompt(messages, max_new_tokens: int = 512):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
        )

    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

    try:
        parsed_response = processor.parse_response(response)
    except Exception:
        parsed_response = response

    return response, parsed_response

raw_response, parsed_response = run_audio_prompt(messages)
parsed_response

{'role': 'assistant',
 'content': 'The audio provided appears to be speech. \n \n**Transcription:**\n"Oh, kabarito today, feeling refreshed. The morning light was beautiful, and I enjoyed a nice cup of coffee."\n \n**Sounds and Confidence:**\nI heard a clear voice speaking a casual, conversational style in English. The overall audio quality is good, allowing for easy comprehension of the spoken words. I am highly confident in this transcription.\n'}

## Silence Direction Probe

The next cells build a balanced sound-vs-silence dataset from the local AudioSet and Speech/Noise datasets, extract Gemma 4 last-token head activations with the same tracing procedure used in `utils.py`, train an L2 logistic probe for each attention head, and report per-head accuracy.

In [4]:
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from datasets import Audio, Dataset, load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

from utils import extract_features, model_base_name, save_features, set_seed

set_seed(42)

AUDIOSET_DIR = Path("datasets/AudioSet")
SPEECH_NOISE_DIR = Path("datasets/speech-noise-dataset")
PROBE_PREFIX = "audio_silence"
PROBE_AUDIO_DIR = Path("data/audio_silence_probe")
PROBE_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

MAX_AUDIOSET_SAMPLES = 32
MAX_SPEECH_NOISE_SAMPLES = 32
MAX_SECONDS = 10.0
N_SPLITS = 5
RIDGE_ALPHA = 1.0

# Keep HF Datasets from writing large caches in home by using streaming reads for parquet.
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")

device = next(model.parameters()).device
results_dir = Path("results") / model_base_name(MODEL_ID)
results_dir.mkdir(parents=True, exist_ok=True)

Set seed 42


In [5]:
def load_local_audioset(max_samples: int):
    iterable = load_dataset(
        "parquet",
        data_files={"train": str(AUDIOSET_DIR / "data" / "bal_train" / "*.parquet")},
        split="train",
        streaming=True,
    )

    selected = []
    for example in iterable:
        human_labels = set(example.get("human_labels") or [])
        if bool(human_labels) and "Silence" not in human_labels:
            selected.append(example)
            if len(selected) >= max_samples:
                break

    if not selected:
        raise RuntimeError("No AudioSet examples matched the non-silence filter.")

    return Dataset.from_list(selected).cast_column("audio", Audio())


def load_local_speech_noise(max_samples: int):
    iterable = load_dataset(
        "parquet",
        data_files={"train": str(SPEECH_NOISE_DIR / "data" / "train-*.parquet")},
        split="train",
        streaming=True,
    )

    selected = []
    for example in iterable:
        selected.append(example)
        if len(selected) >= max_samples:
            break

    if not selected:
        raise RuntimeError("No examples found in speech-noise parquet shards.")

    return Dataset.from_list(selected).cast_column("audio", Audio())


def trim_audio(audio_item, max_seconds: float = MAX_SECONDS):
    array = np.asarray(audio_item["array"], dtype=np.float32)
    if array.ndim > 1:
        array = array.mean(axis=1)
    sample_rate = int(audio_item["sampling_rate"])
    max_len = max(1, int(max_seconds * sample_rate))
    return array[:max_len], sample_rate


def write_probe_pair(array: np.ndarray, sample_rate: int, stem: str):
    sound_path = PROBE_AUDIO_DIR / f"{stem}_sound.wav"
    silence_path = PROBE_AUDIO_DIR / f"{stem}_silence.wav"
    sf.write(sound_path, array, sample_rate)
    sf.write(silence_path, np.zeros_like(array), sample_rate)
    return sound_path, silence_path


audioset_ds = load_local_audioset(MAX_AUDIOSET_SAMPLES)
speech_noise_ds = load_local_speech_noise(MAX_SPEECH_NOISE_SAMPLES)

records = []

for idx, example in enumerate(tqdm(audioset_ds, desc="AudioSet samples")):
    array, sample_rate = trim_audio(example["audio"])
    if array.size == 0 or np.max(np.abs(array)) == 0.0:
        continue
    sound_path, silence_path = write_probe_pair(array, sample_rate, f"audioset_{idx:04d}")
    description = ", ".join(example.get("human_labels") or []) or "AudioSet clip"
    records.extend(
        [
            {"source": "audioset", "audio_path": str(sound_path), "label": 1, "label_name": "sound", "description": description},
            {"source": "audioset", "audio_path": str(silence_path), "label": 0, "label_name": "silence", "description": "Synthetic silence matched to AudioSet clip"},
        ]
    )

for idx, example in enumerate(tqdm(speech_noise_ds, desc="Speech/noise samples")):
    array, sample_rate = trim_audio(example["audio"])
    if array.size == 0 or np.max(np.abs(array)) == 0.0:
        continue
    sound_path, silence_path = write_probe_pair(array, sample_rate, f"speech_noise_{idx:04d}")
    description = str(example.get("label") or "speech-noise clip")
    records.extend(
        [
            {"source": "speech_noise", "audio_path": str(sound_path), "label": 1, "label_name": "sound", "description": description},
            {"source": "speech_noise", "audio_path": str(silence_path), "label": 0, "label_name": "silence", "description": "Synthetic silence matched to speech/noise clip"},
        ]
    )

contrastive_df = pd.DataFrame(records).sample(frac=1.0, random_state=42).reset_index(drop=True)
if contrastive_df.empty:
    raise RuntimeError("No probe examples were created from the local audio datasets.")

display(contrastive_df.head())
display(contrastive_df.groupby(["source", "label_name"]).size().rename("count").reset_index())
print(f"Total examples: {len(contrastive_df)}")

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

AudioSet samples:   0%|          | 0/32 [00:00<?, ?it/s]

Speech/noise samples:   0%|          | 0/32 [00:00<?, ?it/s]

,source,audio_path,label,label_name,description
0,audioset,data/audio_silence_probe/audioset_0027_silence...,0,silence,Synthetic silence matched to AudioSet clip
1,audioset,data/audio_silence_probe/audioset_0020_sound.wav,1,sound,"Organ, Electronic organ"
2,audioset,data/audio_silence_probe/audioset_0009_silence...,0,silence,Synthetic silence matched to AudioSet clip
3,audioset,data/audio_silence_probe/audioset_0015_silence...,0,silence,Synthetic silence matched to AudioSet clip
4,speech_noise,data/audio_silence_probe/speech_noise_0017_sou...,1,sound,clean_speech


,source,label_name,count
0,audioset,silence,32
1,audioset,sound,32
2,speech_noise,silence,32
3,speech_noise,sound,32


Total examples: 128


In [7]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=2560, out_features=512, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
            (v_proj): Linear(in_features=2560, out_features=512, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (down_proj): Linear(in_features=10240, out_features=2560, bias=Fals

In [6]:
PROBE_PROMPT = (
    "Determine whether this audio clip contains audible sound or is silent. "
    "Use the audio content, including faint speech or background noise, to decide."
 )


def build_probe_messages(audio_path: str):
    return [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": audio_path},
                {"type": "text", "text": PROBE_PROMPT},
            ],
        }
    ]


encoded_prompts = [
    processor.apply_chat_template(
        build_probe_messages(audio_path),
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )
    for audio_path in tqdm(contrastive_df["audio_path"], desc="Encoding prompts")
]

labels = contrastive_df["label"].astype(np.int64).tolist()
features = extract_features(
    model=model,
    prompts=encoded_prompts,
    device=device,
    mode="vision",
)
save_features(
    features=features,
    labels=labels,
    model_path=MODEL_ID,
    output_dir="results",
    prefix=PROBE_PREFIX,
    save_as_numpy=False,
 )

print(f"Feature tensor shape: {features.shape}")

Encoding prompts:   0%|          | 0/128 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/128 [00:00<?, ?it/s]

LookupError: model.language_model.layers.0.self_attn.head_out

In [ ]:
def train_logistic_head_probes(
    features: np.ndarray,
    labels: list[int],
    alpha: float = RIDGE_ALPHA,
    n_splits: int = N_SPLITS,
    seed: int = 42,
 ):
    labels = np.asarray(labels, dtype=np.int64)
    _, _, n_layers, n_heads, _ = features.shape
    performance = np.zeros((n_layers, n_heads), dtype=np.float32)
    probe_models = {}
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    c_value = 1.0 / alpha if alpha > 0 else 1e6

    for layer_idx in tqdm(range(n_layers), desc="Training logistic probes"):
        probe_models[layer_idx] = {}
        for head_idx in range(n_heads):
            X = features[:, 0, layer_idx, head_idx, :]
            fold_scores = []

            for train_idx, test_idx in splitter.split(X, labels):
                clf = LogisticRegression(
                    penalty="l2",
                    C=c_value,
                    fit_intercept=False,
                    solver="liblinear",
                    max_iter=2000,
                )
                clf.fit(X[train_idx], labels[train_idx])
                preds = clf.predict(X[test_idx])
                fold_scores.append(accuracy_score(labels[test_idx], preds))

            performance[layer_idx, head_idx] = float(np.mean(fold_scores))

            final_clf = LogisticRegression(
                penalty="l2",
                C=c_value,
                fit_intercept=False,
                solver="liblinear",
                max_iter=2000,
            )
            final_clf.fit(X, labels)
            probe_models[layer_idx][head_idx] = final_clf

    return performance, probe_models


silence_performance, silence_probe_models = train_logistic_head_probes(
    features=features,
    labels=labels,
    alpha=RIDGE_ALPHA,
    n_splits=N_SPLITS,
    seed=42,
 )

with open(results_dir / f"{PROBE_PREFIX}_logreg_performance.pkl", "wb") as handle:
    pickle.dump(silence_performance, handle)
with open(results_dir / f"{PROBE_PREFIX}_logreg.pkl", "wb") as handle:
    pickle.dump(silence_probe_models, handle)

accuracy_df = (
    pd.DataFrame(silence_performance)
    .rename_axis(index="layer", columns="head")
    .stack()
    .rename("accuracy")
    .reset_index()
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
 )
accuracy_df["accuracy_percent"] = accuracy_df["accuracy"] * 100.0

best_layer = int(accuracy_df.loc[0, "layer"])
best_head = int(accuracy_df.loc[0, "head"])
sound_direction = silence_probe_models[best_layer][best_head].coef_.astype(np.float32).copy()
silence_direction = -sound_direction

plt.figure(figsize=(12, 8), dpi=150)
sns.heatmap(silence_performance * 100.0, cmap="viridis", vmin=0, vmax=100)
plt.title("Gemma 4 sound vs silence accuracy by head")
plt.xlabel("Head")
plt.ylabel("Layer")
plt.tight_layout()
plt.show()

display(accuracy_df.head(20))
print(f"Best head: layer {best_layer}, head {best_head}, accuracy={accuracy_df.loc[0, 'accuracy']:.3f}")
print(f"Silence direction shape: {silence_direction.shape}")